# 10 — The kill-test and its verdict

The Phase-0 charter's existence question: **does signed-net-flux (Target A)
survive the QSE cancellation that the literature has warned about since
Hix & Thielemann 1996?** If net flow hides under massive forward/reverse
cancellation, predicting φ and mapping dY = νφ buys exact conservation and
loses the accuracy of the headline quantity.

> **VERDICT (2026-07-12): TARGET A — full-width flux head; the hybrid's
> equilibrium mask is measured EMPTY on the relaxed label manifold. No
> Target-B switch condition fires.**
> (`docs/phase0-killtest-verdict.md`)

This notebook **displays** that verdict and re-derives a *quick-look* of the
distributions behind it. `scripts/run_killtest.py` prints to stdout and
persists nothing, so the exact verdict numbers are quoted from RESULTS.md /
the verdict doc rather than recomputed — the capped recomputation here (120
rows from shipped trajectories vs 4,693 rows from 229 trajectories/net) is
illustrative and is stamped as such.

All κ work below is **UNSCREENED** (the two-κ rule).

In [ ]:
import sys
from pathlib import Path

_here = Path.cwd()
if not (_here / "nbsupport.py").exists():
    _root = next(p for p in [_here, *_here.parents] if (p / "pyproject.toml").exists())
    _here = _root / "notebooks" / "phase0"
sys.path.insert(0, str(_here))

import matplotlib.pyplot as plt
import numpy as np

import nbsupport as nbs
from gnn_nucleo.killtest.active_set import EPS_SWEEP, cond_s_active, guidry_masks
from gnn_nucleo.killtest.distributions import kappa_summary
from gnn_nucleo.killtest.manifold import assemble
from gnn_nucleo.killtest.strata import assign_strata

nbs.style()
QUICK = nbs.QUICK
NET = "mesa_80"
MAX_ROWS = 6 if QUICK else 24
RUN_IDS = ["trajectories-unscreened"] + ([] if QUICK else ["rerun-trajectories-unscreened"])

In [ ]:
nbs.provenance_header(
    "10",
    "Kill-test distributions + the verdict",
    nbs.status_of([6, 7, 8, 9]),
    results_rows=[
        "2026-07-12: VERDICT — TARGET A, full-width flux head, no Guidry mask deployed (docs/phase0-killtest-verdict.md)",
        "2026-07-12: relaxed manifold — frac κ>0.1 falls to 0.70/0.64 at T9 [5.0,6.3); κ-balanced (κ<1e-3) ≤ 0.4 % of carrying pairs everywhere (a cancellation CONTINUUM)",
        "2026-07-12: Guidry maskable set EMPTY — median AND p90 = 0 columns per row at every ε, both nets",
        "2026-07-12: cond(S_active) = 41.8 / 57.4 (mesa_80/151) at every ε ≪ the 1e6 gate",
        "2026-07-12: |dẎₑ| coverage = 1.0000 (weak κ = 1.0 verified on 8.8M / 33.2M samples); dominant-isotope coverage SPLIT (median ≥95 % except [5.0,6.3): 0.62–0.80)",
    ],
    data=[
        "data/fluxes/{net}/subsample-unscreened (training grid)",
        "data/fluxes/{net}/trajectories-unscreened (+ rerun-… at NB_QUICK=0) — the relaxed manifold",
        "docs/phase0-killtest-verdict.md (the rendered verdict)",
    ],
    scripts=["scripts/run_killtest.py --training-grid / --relaxed --include-reruns"],
)

## The two distributions

The kill-test's central methodological point: **which distribution you ask
matters.** The training grid (Sobol-random compositions) says everything is
active — a vacuous pass. The relaxed manifold (states that have actually
burned) is structured. The verdict is rendered on the latter.

In [ ]:
grid = kappa_summary(nbs.flux_store(NET, "subsample-unscreened"))
print(f"training grid ({NET}):")
for r in grid:
    print(f"  T9 {r['stratum']:>11}: frac κ>0.1 = {r['frac_gt_0p1']:.3f}  "
          f"frac κ<1e-3 = {r['frac_lt_1em3']:.3f}  n = {r['n_samples']:,}")

rows = assemble(NET, RUN_IDS, prestall=True, max_rows_per_traj=MAX_ROWS, with_delta=True)
t9b, yeb = assign_strata(rows.T, rows.ye)
print(f"\nrelaxed manifold ({NET}): {rows.n_rows} rows from "
      f"{len(set(rows.traj_key))} trajectories, NSE converged on {int(rows.nse_converged.sum())}")

## Figure 1 — κ by stratum: training grid vs relaxed manifold

Both panels are UNSCREENED — one convention, so one figure is legal here.

In [ ]:
z = nbs.load_nu(NET)
nu, weak = z["nu"], z["weak_mask"].astype(bool)
from gnn_nucleo.fluxes.store import FluxStore

c0 = next(iter(FluxStore(NET, RUN_IDS[0]).iter_chunks()))
strong_fwd = c0.is_forward_member & (c0.pair_col >= 0)
net_cols = c0.is_forward_member | (c0.pair_col < 0)

# relaxed frac κ>0.1 per stratum, carrying strong pairs (run_killtest convention)
rel = {}
for b in range(len(nbs.T9_LABELS)):
    m = np.nonzero((t9b == b) & rows.nse_converged)[0]
    if m.size == 0:
        continue
    fr = []
    for r in m:
        f = rows.f_plus[strong_fwd, r]
        k = rows.kappa[strong_fwd, r]
        pos = f[f > 0]
        if pos.size == 0:
            continue
        sel = f > np.median(pos)
        if sel.sum():
            fr.append((k[sel] > 0.1).mean())
    if fr:
        rel[nbs.T9_LABELS[b]] = float(np.median(fr))

fig, ax = plt.subplots(figsize=(10.5, 4.6))
labs = nbs.T9_LABELS
xg = np.arange(len(labs))
gd = {r["stratum"]: r["frac_gt_0p1"] for r in grid}
ax.bar(xg - 0.2, [gd.get(x, np.nan) for x in labs], 0.4, label="training grid (Sobol states)",
       color="#0072B2")
ax.bar(xg + 0.2, [rel.get(x, np.nan) for x in labs], 0.4, label="relaxed manifold (burned states)",
       color="#D55E00")
ax.axhline(0.95, color="#333333", ls="--", lw=1.2, label="Target-A viability line (≥95 %)")
ax.set_xticks(xg, labs)
ax.set_ylim(0, 1.05)
ax.set_xlabel("T₉ stratum")
ax.set_ylabel("frac of carrying pairs with κ > 0.1")
ax.set_title(f"{NET} — κ active fraction · UNSCREENED · the distribution matters")
ax.legend(fontsize=8, loc="lower left")
if QUICK:
    nbs.quick_banner(fig)
nbs.caption(
    fig,
    "The training grid says ~everything is active everywhere; the relaxed manifold does not — the "
    "measured dip is to frac κ>0.1 = 0.70/0.64 (mesa_80/151) at T9 [5.0,6.3). But κ-balanced "
    "(κ<1e-3) pairs stay ≤ 0.4 % of carrying pairs everywhere: this is a cancellation CONTINUUM, "
    "not a clean equilibrated sector that a mask could cleanly remove. Quick-look shares here "
    "(120 rows, shipped trajectories); the verdict used 4,693 rows/net from 229 trajectories.",
    results=[
        "RESULTS.md 2026-07-12 relaxed-manifold rows (frac κ>0.1 → 0.70/0.64 at [5.0,6.3); κ<1e-3 ≤ 0.4 %)",
        "RESULTS.md 2026-07-11/12 training-grid rows (0.985–0.999 / 0.974–0.997)",
    ],
    scripts=["scripts/run_killtest.py --training-grid / --relaxed --include-reruns"],
)

## Figure 2 — the Guidry mask is EMPTY

The planned hybrid architecture would mask equilibrated columns using the
Guidry criterion |y − ȳ|/ȳ < ε, referenced to true (Saha) NSE, with an ε
sweep of {3e-3, 1e-2, 3e-2}. **Weak columns are never eligible** — that
exclusion is structural in `qse.diagnostics.eligible_mask` (invariant #2),
not a convention.

The measured answer: zero maskable columns per row, at every ε, both
networks — median *and* p90. The reason is notebook 09's finding: the
manifold's own equilibria are bug-displaced, so δ_r referenced to *true* NSE
effectively never crosses threshold.

In [ ]:
counts = []
for r in np.nonzero(rows.nse_converged)[0]:
    m = guidry_masks(rows.delta_r[:, r], weak)
    counts.append([int(m[e].sum()) for e in EPS_SWEEP])
counts = np.array(counts)
for i, e in enumerate(EPS_SWEEP):
    print(f"  ε = {e:>6}: maskable columns per row — median {np.median(counts[:, i]):.1f}, "
          f"p90 {np.percentile(counts[:, i], 90):.1f}, max {counts[:, i].max()}")

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.4), width_ratios=[1.2, 1])
ax = axes[0]
x = np.arange(len(EPS_SWEEP))
ax.bar(x - 0.2, [np.median(counts[:, i]) for i in range(len(EPS_SWEEP))], 0.4, label="median")
ax.bar(x + 0.2, [np.percentile(counts[:, i], 90) for i in range(len(EPS_SWEEP))], 0.4, label="p90")
ax.set_xticks(x, [f"ε = {e}" for e in EPS_SWEEP])
ax.set_ylabel("maskable columns per row")
ax.set_ylim(0, max(1.0, counts.max() * 1.2))
ax.set_title("Guidry maskable set (weak columns structurally excluded)")
ax.text(0.5, 0.55, "EMPTY at every ε", transform=ax.transAxes, ha="center",
        fontsize=15, fontweight="bold", color="#D55E00")
ax.legend(fontsize=8)

ax = axes[1]
ok = rows.nse_converged
d = np.log10(np.maximum(rows.delta_r[:, ok][~weak], 1e-16)).ravel()
ax.hist(d[np.isfinite(d)], bins=60, color="#0072B2")
for e in EPS_SWEEP:
    ax.axvline(np.log10(e), color="#D55E00", ls="--", lw=1)
ax.text(np.log10(3e-2) + 0.3, ax.get_ylim()[1] * 0.7, "ε sweep\n(nothing to the left)",
        fontsize=8, color="#D55E00")
ax.set_xlabel("log₁₀ δ_r  (departure from TRUE NSE, non-weak columns)")
ax.set_ylabel("column × row samples")
ax.set_title("Why: δ_r never gets near the threshold")
if QUICK:
    nbs.quick_banner(fig)
nbs.caption(
    fig,
    "Mask design is settled by EMPTINESS, not by tuning: no Guidry mask is deployed, and κ "
    "diagnostics are carried as FEATURES instead. Churn (0.00–0.03 flips/step, ≤ 0.002 %/step) is "
    "far below the 5 %/step freeze trigger and moot given emptiness. The ε≈0.01 transfer "
    "hypothesis is moot on this data. Because the mask is empty, S_active = all net columns and "
    "cond(S_active) = cond(ν net columns) = 41.8/57.4 — nowhere near the 1e6 Target-B switch.",
    results=[
        "RESULTS.md 2026-07-12 Guidry ε-sweep rows (median AND p90 = 0 at every ε, both nets)",
        "RESULTS.md 2026-07-12 churn rows (0.00–0.03 flips/step, n = 35 QSE-window trajectories)",
        "RESULTS.md 2026-07-11 label-pathology rows (the manifold's equilibria are displaced)",
    ],
    scripts=["scripts/run_killtest.py --relaxed --include-reruns"],
)

## Figure 3 — cond(S_active): no Target-B switch

The ADR-0001 switch condition is cond(S_active) > 1e6. With the mask empty,
S_active is the full set of net columns and the conditioning question
collapses to cond(ν)'s — three to four orders of magnitude inside the gate.

In [ ]:
# cond_s_active takes a BOOLEAN column mask (same call shape as scripts/run_killtest.py).
cond_now = cond_s_active(nu, net_cols)
cond_full = cond_s_active(nu, np.ones(nu.shape[1], dtype=bool))
print(f"{NET}: cond(S_active) with the empty mask (S_active = all {int(net_cols.sum())} "
      f"net columns) = {cond_now:.1f};  cond(ν full) = {cond_full:.1f}")

fig, ax = plt.subplots(figsize=(8.5, 4.2))
ax.bar(["mesa_80\n(measured)", "mesa_151\n(measured)"], [41.8, 57.4], color="#009E73", width=0.5)
ax.axhline(1e6, color="#E69F00", ls="--", lw=1.5, label="Target A→B switch: cond > 1e6")
ax.axhline(1e8, color="#D55E00", ls="--", lw=1.5, label="kill-test FAIL: cond > 1e8")
ax.set_yscale("log")
ax.set_ylim(1, 1e9)
ax.set_ylabel("cond(S_active)")
ax.set_title("Active-set conditioning vs the switch/fail gates (rank-revealing definition)")
ax.legend(fontsize=8)
ax.text(0.5, 0.18, "~4 orders of margin", transform=ax.transAxes, ha="center",
        fontsize=11, color="#005f45", fontweight="bold")
nbs.caption(
    fig,
    f"Measured 41.8 / 57.4 (mesa_80/151) at every ε, on BOTH the training grid and the relaxed "
    f"manifold (this notebook's live recomputation on the empty-mask active set: {cond_now:.1f}, "
    f"full ν {cond_full:.1f} vs the measured 41.6/57.9). "
    "Conservation left-null vectors are structural, hence the rank-revealing definition (ν "
    "restricted to net columns). No conditioning-based reason to abandon Target A exists.",
    results=[
        "RESULTS.md 2026-07-12 cond(S_active) rows (41.8/57.4; full ν 41.6/57.9; checklist row 7)",
        "docs/decisions/0001 (Target-A default + its switch conditions)",
    ],
    scripts=["scripts/run_killtest.py", "scripts/graph_metrics.py"],
)

## Figure 4 — the verdict dashboard

All eight thresholds from `docs/phase0-killtest-verdict.md`, with their
measured outcomes. **Quoted, not recomputed** — these are the citable
numbers.

In [ ]:
# Verbatim from docs/phase0-killtest-verdict.md §Thresholds (2026-07-12).
THRESHOLDS = [
    ("1. ≥95 % of |ΔYₑ| carried by {κ>0.1}", "PASS", "1.0000 — weak κ = 1.0 verified on 8.8M/33.2M samples;\ntop-5 weak channels carry 0.96/0.75 (⁵⁶Ni EC #1)"),
    ("2. ≥95 % of dominant-isotope |ΔX| carried", "SPLIT", "median species ≥95 % except T9 [5.0,6.3) (0.62–0.80);\nworst species/row 0.002–0.17 across [4.0,6.3)"),
    ("3. FAIL if spread >~30 % of reactions near κ floor", "PASS", "per-row low-κ share median 0.9 %–24 % in every\nstratum × phase — no diffuse-spread failure"),
    ("4. cond(S_active) < 1e6 pass / > 1e8 fail", "PASS", "41.8 / 57.4 (mask empty ⇒ S_active = all net columns)"),
    ("5. Guidry ε sweep {3e-3, 1e-2, 3e-2}", "EMPTY", "median AND p90 = 0 maskable columns at every ε,\nboth nets ⇒ no mask deployed"),
    ("6. Mask churn < 5 %/step freeze trigger", "PASS", "0.00–0.03 flips/step (≤0.002 %/step), n = 35\nQSE-window trajectories — moot given (5)"),
    ("7. Timescale separation ('6–8 orders')", "RETIRED", "measured −12.2…+1.4 dex medians (p10 −25, p90 +3.5):\nno separated fast sector exists"),
    ("8. Energy consistency ≤ 1 % (invariant #5)", "PASS*", "same-fluxes gate: frac ≤1 % = 1.000 in every MEASURED\nstratum (T9 ≥ 5 censored). *independent leg 0.979/0.996 —\nmesa_80 at 2.1 %, OPEN"),
]
STATUS_C = {"PASS": "#009E73", "SPLIT": "#E69F00", "EMPTY": "#0072B2",
            "RETIRED": "#CC79A7", "PASS*": "#8FBF9F"}

fig, ax = plt.subplots(figsize=(13.5, 6.2))
ax.axis("off")
for i, (name, status, detail) in enumerate(THRESHOLDS):
    y = len(THRESHOLDS) - i
    ax.add_patch(plt.Rectangle((0, y - 0.42), 13.2, 0.84, fc="#F7F7F7", ec="#DDDDDD"))
    ax.text(0.1, y, name, va="center", fontsize=9, fontweight="bold")
    ax.add_patch(plt.Rectangle((5.0, y - 0.28), 1.15, 0.56, fc=STATUS_C[status], ec="none"))
    ax.text(5.575, y, status, va="center", ha="center", fontsize=8.5,
            color="white", fontweight="bold")
    ax.text(6.4, y, detail, va="center", fontsize=7.5, color="#333333")
ax.set_xlim(0, 13.4)
ax.set_ylim(0.3, len(THRESHOLDS) + 0.9)
ax.text(0.1, len(THRESHOLDS) + 0.6,
        "VERDICT: TARGET A — full-width flux head, equilibrium mask measured EMPTY, "
        "no Target-B switch fires",
        fontsize=11.5, fontweight="bold", color="#005f45")
nbs.caption(
    fig,
    "Quoted from the verdict doc — these are the citable numbers, not recomputed here. Two carry "
    "forward: threshold 2's SPLIT (a few dominant species' net evolution rides on near-cancelled "
    "columns with 1/κ error amplification — mitigated by Φ-level supervision + loss weighting, NOT "
    "a target switch, since Target B faces the same cancelled sums inside its dY labels) and "
    "threshold 8's open 2.1 % mesa_80 margin. The conditional band splits: Target A is "
    "UNCONDITIONAL below T9 4.0; for [4.0,5.0) worst-species coverage fails the 95 % bar but "
    "the mitigation is MEASURED-FEASIBLE (local Φ labels are proven for T9 < 5); only for "
    "[5.0,6.3) — where median-species coverage also fails and the label physics is "
    "bug-displaced — is supervision design DEFERRED to the benchmark-vs-physics decision "
    "(notebook 09).",
    results=[
        "docs/phase0-killtest-verdict.md §Thresholds (2026-07-12) — all eight rows",
        "RESULTS.md 2026-07-12 kill-test rows",
    ],
    scripts=["scripts/run_killtest.py", "scripts/step6_integrate_check.py", "scripts/step6_eps_pin.py"],
)

## What this notebook does NOT show

- **The full-depth distributions.** `NB_QUICK=0` adds the 209 rerun
  trajectories/net and 24 rows each (tens of minutes). Even then, the
  citable numbers remain the RESULTS.md rows: `run_killtest.py` persists
  nothing, so a future `--dump-artifacts` flag would be the right way to
  plot the exact verdict distributions here.
- κ↔δ correlation (+0.491/+0.498), phase splits (pre-arrival vs attractor),
  and the per-stratum coverage tables — `scripts/run_killtest.py` stdout.
- Why the mask is empty (the displaced equilibria): notebook 09.